# SVU–KJSCE Django and Python Laboratory

TY B Tech COMP BoS · 2023_1.0 · pp. 80–81

Run with the repository virtual environment as your kernel. Install `ipykernel` if using Jupyter. This notebook uses a disposable database and does not modify course.sqlite3. Terminal and browser tasks remain explicit exercises. See docs/syllabus-and-lab-record.md for the complete record and rubric.

In [ ]:
from pathlib import Path
import os, sys, tempfile
ROOT = Path.cwd()
if not (ROOT / "manage.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "manage.py").exists(), "Open notebook from the repository or notebooks folder"
sys.path.insert(0, str(ROOT))
workspace = tempfile.TemporaryDirectory()
os.environ["DJANGO_SETTINGS_MODULE"] = "gitlab_project.settings"
os.environ["DJANGO_DB_PATH"] = str(Path(workspace.name) / "notebook.sqlite3")
import django
django.setup()
from django.core.management import call_command
call_command("migrate", verbosity=0)
print("Django", django.get_version(), "— temporary database ready")

## Unit 1.1: Django project and development environment — CO1

Django is a Python web framework with URL routing, templates, an ORM, forms, authentication and an administration site. A project contains site configuration; an app contains a reusable feature. Use a supported Python version and isolate dependencies in a virtual environment. This repository uses the Django 5.2 LTS series; requirements.lock records the tested versions.

1. Create and activate a virtual environment with Python 3.13.
2. Install requirements and inspect pip show Django.
3. Inspect manage.py, settings.py, urls.py and wsgi.py.
4. Run migrations, start the development server and open /health/.
5. Create a separate practice project using startproject; compare its generated structure.

Browser practical: `/health/`

Study commands (run shell examples in a terminal):
```text
python3.13 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.lock
python -m django --version
python -m django startproject practice_project
python manage.py migrate
python manage.py runserver
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.conf import settings
assert "course" in settings.INSTALLED_APPS
print(settings.ROOT_URLCONF, settings.AUTH_USER_MODEL)

## Unit 1.2: JavaScript, JSX and verbatim templates — CO1

Django renders HTML on the server. React, Vue and other JavaScript libraries can update selected regions in the browser. JSX is syntax compiled into JavaScript calls; browsers do not execute raw JSX. Django template delimiters may collide with client template syntax. The verbatim tag preserves a region literally. This exercise uses a local small ES module; a separate JSX source and its equivalent compiled calls show the React boundary without claiming vanilla JavaScript is JSX.

1. Open the template laboratory.
2. Change the name query parameter and inspect server-rendered output.
3. Increment the browser counter; reload and observe its reset.
4. Read the JSX and equivalent React.createElement examples.
5. Inspect the literal {{ client_value }} output inside a verbatim block.

Browser practical: `/course/templates/`

Study commands (run shell examples in a terminal):
```text
{% verbatim %}
<p>{{ client_value }}</p>
{% endverbatim %}

// JSX (compile with a JSX toolchain):
const element = <h3>Hello, {name}</h3>;
// Equivalent React call:
const element = React.createElement("h3", null, "Hello, ", name);
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.template import Template, Context
assert Template("{% verbatim %}{{ client_value }}{% endverbatim %}").render(Context()) == "{{ client_value }}"
print("Client delimiters preserved")

## Unit 1.3: Git repositories, remotes, branches and merges — CO1

Git stores snapshots in commits and names lines of development with branches. A remote is a named repository URL; fetch downloads refs and push publishes commits. GitHub is a hosted Git server. A local bare repository demonstrates the server side without creating an external account. A production SSH Git server additionally needs user access control, SSH keys, backups and protected refs. Resolve conflicts in the working tree before committing the merge.

1. Use the existing Git experiment for branch, merge and conflict practice.
2. Run the commands below in a disposable directory, not this checkout.
3. Clone the bare remote into a second working directory.
4. Create a feature branch, commit a file and push it.
5. Fetch from the second clone, merge the branch and inspect git log --graph --all.

Browser practical: `/`

Study commands (run shell examples in a terminal):
```text
mkdir git-practice && cd git-practice
git init --bare classroom.git
git clone classroom.git student
cd student
git switch -c main
echo "Lab record" > record.txt
git add record.txt
git commit -m "Add lab record"
git push -u origin main
git switch -c feature
echo "Observation" >> record.txt
git commit -am "Record observation"
git switch main
git merge feature
git push origin main
git remote -v
```

Record your actual observation and explain one failure case before continuing.

## Unit 2.1: URL mapping, function views and class views — CO2

A URL pattern maps a request path to a callable. A function-based view receives HttpRequest and returns HttpResponse. A class-based view uses as_view() to expose a callable and dispatches HTTP methods to methods such as get(). Named routes let templates use reverse lookup instead of hardcoded paths. Django templates are escaped by default; keep user input escaped.

1. Inspect course/urls.py and the experiment function in course/views.py.
2. Open /course/templates/ to run the TemplateView subclass.
3. Change ?name=Asha and inspect the response.
4. Request an unknown experiment slug and confirm HTTP 404.
5. Compare a function view with TemplateLab.get_context_data.

Browser practical: `/course/templates/`

Study commands (run shell examples in a terminal):
```text
path("templates/", TemplateLab.as_view(), name="templates")

class TemplateLab(TemplateView):
    template_name = "course/template_lab.html"

# Function view:
def index(request):
    return render(request, "course/index.html", context)
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.urls import resolve, reverse
print(resolve("/course/templates/").func.view_class.__name__)
assert reverse("course:observations") == "/course/observations/"

## Unit 2.2: Context variables, loops and template filters — CO2

A view supplies a context dictionary to render(). A template reads keys as variables, loops over collections and applies filters using a pipe. The upper filter changes presentation, while length counts elements. Template inheritance keeps the header and navigation consistent. Template rendering is not a place for database writes or arbitrary Python execution.

1. Open the template laboratory with your name in the query string.
2. Compare the original name and its uppercase form.
3. Inspect the three measurements rendered with a for loop.
4. Enter HTML in the name query parameter and verify it appears as text.
5. Inspect base.html, extends and block tags.

Browser practical: `/course/templates/`

Study commands (run shell examples in a terminal):
```text
context = {"name": "Asha", "values": [4, 8, 12]}
return render(request, "course/template_lab.html", context)

{{ name|upper }}
{{ values|length }}
{% for value in values %}<li>{{ value }}</li>{% endfor %}
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.test import Client
client = Client()
response = client.get("/course/templates/", {"name":"Asha"})
assert response.status_code == 200
assert b"ASHA" in response.content
print("Context and upper filter verified")

## Unit 2.3: Static files, finders and static URLs — CO2

Static files include CSS, JavaScript and images. AppDirectoriesFinder searches installed apps under static/; FileSystemFinder searches STATICFILES_DIRS when configured. The static template tag constructs a static URL using the configured storage. findstatic reports where a file is found. collectstatic gathers deployment assets into STATIC_ROOT. Development runserver serving is not a production static server.

1. Inspect lab/static/lab/course.css and template static tags.
2. Run findstatic with verbosity 2 and identify the winning file.
3. Inspect the default STATICFILES_FINDERS in the Django shell.
4. Run collectstatic and inspect staticfiles/.
5. For a FileSystemFinder exercise, add a temporary directory to STATICFILES_DIRS and repeat findstatic.

Browser practical: `/course/templates/`

Study commands (run shell examples in a terminal):
```text
python manage.py findstatic lab/course.css --verbosity 2
python manage.py collectstatic --noinput
python manage.py shell -c "from django.conf import settings; print(settings.STATICFILES_FINDERS)"

{% load static %}
<link rel="stylesheet" href="{% static 'lab/course.css' %}">
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.contrib.staticfiles import finders
asset = finders.find("lab/course.css")
assert asset
print(Path(asset).relative_to(ROOT))

## Unit 3.1: Forms, validation and cleaned Python values — CO3

A bound form contains submitted data. is_valid() runs field conversion, field validators, clean_field methods and form-wide clean(). cleaned_data contains Python values such as Decimal rather than raw request strings. Initial values apply to unbound forms; placeholders are hints, not stored values. POST requests need CSRF protection and successful writes redirect to prevent duplicate submissions on refresh.

1. Register and sign in.
2. Submit a title shorter than three characters and a negative value.
3. Read both field errors and confirm no row was created.
4. Submit a valid measurement and refresh after the redirect.
5. Edit the row and compare the placeholder with its initial value.

Browser practical: `/course/observations/`

Study commands (run shell examples in a terminal):
```text
class ObservationForm(forms.ModelForm):
    def clean_value(self):
        value = self.cleaned_data["value"]
        if value < 0:
            raise forms.ValidationError("Use a non-negative value.")
        return value

if request.method == "POST" and form.is_valid():
    observation = form.save(commit=False)
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from course.forms import ObservationForm
bad = ObservationForm({"title":"x", "value":"-1"})
assert not bad.is_valid()
good = ObservationForm({"title":"Latency", "value":"12.50"})
assert good.is_valid()
print(type(good.cleaned_data["value"]).__name__, dict(bad.errors))

## Unit 3.2: CSV processing, graphs and binary responses — CO3

Python csv.DictReader maps column headings to row dictionaries. Validate each row before writing a batch so errors do not leave a partial import. The practical accepts UTF-8 title,value,notes columns and displays measurements as a server-generated horizontal graph. Its labels and numeric table remain available without JavaScript. CSV export uses csv.writer. The self-learning extension streams a real PDF with FileResponse and ReportLab; binary files require byte buffers, MIME types and download headers.

1. Download docs/sample-observations.csv from the repository.
2. Import it while signed in and inspect the graph and table.
3. Try an invalid negative value; verify the entire batch is rejected.
4. Download CSV and PDF and inspect their contents.
5. Compare StringIO for CSV with BytesIO for PDF generation.

Browser practical: `/course/observations/`

Study commands (run shell examples in a terminal):
```text
import csv, io
reader = csv.DictReader(io.StringIO(text))
for row in reader:
    print(row["title"], row["value"])

# Binary download:
return FileResponse(buffer, as_attachment=True, filename="observations.pdf")
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from course.models import User, Observation
from course.services import import_csv
from django.core.files.uploadedfile import SimpleUploadedFile
student = User.objects.create_user("notebook", "student@example.com", "NotebookPass!789")
data = (ROOT / "docs/sample-observations.csv").read_bytes()
count = import_csv(SimpleUploadedFile("sample.csv", data), student)
assert count == 3
print(list(Observation.objects.values_list("title", "value")))

## Unit 3.3: Models, migrations, CRUD and bulk operations — CO3

A model defines database fields and relations. makemigrations creates a versioned schema change; migrate applies it. The ORM translates query operations to parameterized SQL. create, filter, update and delete correspond to INSERT, SELECT, UPDATE and DELETE. bulk_create and bulk_update reduce database round trips but do not call each object save() or emit its save signals; validate explicitly. Use transactions for related writes.

1. Run showmigrations and inspect course/migrations/0001_initial.py.
2. Create, edit and delete your observations in the practical.
3. Run the ORM commands in a disposable database or with test records.
4. Inspect a QuerySet.query and compare it with the SQL examples.
5. Read services.import_csv for the validated bulk_create path and run the notebook bulk_update exercise.

Browser practical: `/course/observations/`

Study commands (run shell examples in a terminal):
```text
Observation.objects.create(owner=user, title="Latency", value=12)
rows = list(Observation.objects.filter(owner=user))
for row in rows:
    row.notes = "Reviewed"
Observation.objects.bulk_update(rows, ["notes"])
Observation.objects.filter(owner=user, title="Latency").delete()

-- Equivalent CRUD concepts (values should be parameters):
INSERT INTO course_observation (owner_id,title,value,notes,created_at) VALUES (1,'Latency',12,'',CURRENT_TIMESTAMP);
SELECT * FROM course_observation WHERE owner_id=1;
UPDATE course_observation SET notes='Reviewed' WHERE owner_id=1;
DELETE FROM course_observation WHERE owner_id=1;
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
rows = list(Observation.objects.filter(owner=student))
for row in rows:
    row.notes = "Reviewed in notebook"
Observation.objects.bulk_update(rows, ["notes"])
assert Observation.objects.filter(notes="Reviewed in notebook").count() == 3
from django.db import connection
with connection.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM course_observation WHERE owner_id = %s", [student.pk])
    assert cursor.fetchone()[0] == 3
print("ORM bulk update and SQL query agree")

## Unit 4.1: Users, sessions, permissions and groups — CO4

Authentication identifies a user; authorization determines allowed actions. This project defines a custom AbstractUser before its first migration and stores roll_number. Django hashes passwords, rotates session credentials at login, and supports password change and token-based reset. The development email backend prints reset messages to the server console. Each observation belongs to its creator. The instructor export requires an explicit permission; staff access and model permissions are separate concepts.

1. Register two users and create an observation for each.
2. Verify each account sees only its own rows.
3. Change a password, then request a password reset using your registered email.
4. Follow the reset URL printed by the console backend; a token is single-use.
5. Create an Instructors group in admin, assign export_observation and test /course/instructor/export/ with and without it.
6. Use the POST logout button and verify protected routes redirect to login.

Browser practical: `/course/register/`

Study commands (run shell examples in a terminal):
```text
from django.contrib.auth.models import Group, Permission
group, _ = Group.objects.get_or_create(name="Instructors")
permission = Permission.objects.get(codename="export_observation", content_type__app_label="course")
group.permissions.add(permission)
user.groups.add(group)
user.has_perm("course.export_observation")
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.contrib.auth.models import Group, Permission
group, _ = Group.objects.get_or_create(name="Instructors")
group.permissions.add(Permission.objects.get(codename="export_observation"))
student.groups.add(group)
assert student.has_perm("course.export_observation")
print("Group permission granted")

## Unit 4.2: Django administration, filters and actions — CO4

Django admin is a trusted staff interface generated from model metadata. Registering a ModelAdmin controls list columns, search fields, filters and actions. This lab customizes the site headings and user fieldsets, adds owner/date filters and a clear-notes action. Staff users still need model permissions. Admin actions can change many records: inspect the selection before confirming an operation.

1. Create a superuser from the terminal; never commit credentials.
2. Open /admin/ and inspect Users, Groups and Observations.
3. Search observations by title or username and filter by owner.
4. Select test observations and run Clear notes.
5. Create a staff account with view-only permissions and compare its controls.

Browser practical: `/admin/`

Study commands (run shell examples in a terminal):
```text
python manage.py createsuperuser

@admin.register(Observation)
class ObservationAdmin(admin.ModelAdmin):
    list_display = ("title", "owner", "value", "created_at")
    list_filter = ("created_at", "owner")
    search_fields = ("title", "owner__username")
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from django.contrib import admin
from course.admin import clear_notes
assert Observation in admin.site._registry
clear_notes(None, None, Observation.objects.filter(owner=student))
assert not Observation.objects.exclude(notes="").exists()
print("Model registered; admin action updated selected records")

## Unit 5.1: REST endpoints and Pydantic validation — CO5

REST uses resource URLs and HTTP semantics: GET reads, POST creates (201), PUT replaces, DELETE removes (204), and missing resources return 404. JSON is a representation format. FastAPI builds on ASGI with type-driven validation and OpenAPI documentation. It supports async functions for awaitable I/O; async alone does not accelerate CPU work or blocking database calls. Django REST Framework integrates closely with Django ORM/auth/admin; FastAPI suits a separate typed API. Neither is universally superior. The companion API uses SQLite persistence and Pydantic constraints, not a process-local dictionary.

1. Start the separate FastAPI service on port 8001.
2. Open /docs and inspect the generated schemas.
3. POST an observation, then GET it by its returned integer ID.
4. Filter the collection with the min_value query parameter.
5. Try a short title or negative value and inspect HTTP 422.
6. Replace and delete the resource; verify a subsequent GET returns 404.

Browser practical: `http://127.0.0.1:8001/docs`

Study commands (run shell examples in a terminal):
```text
uvicorn api.main:app --port 8001

curl -X POST http://127.0.0.1:8001/observations -H "Content-Type: application/json" -d '{"title":"Latency","value":12}'
curl "http://127.0.0.1:8001/observations?min_value=10"
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
from api import main as api
from fastapi.testclient import TestClient
api.DB_PATH = Path(workspace.name) / "api.sqlite3"
with TestClient(api.app) as api_client:
    created = api_client.post("/observations", json={"title":"Latency", "value":12})
    assert created.status_code == 201
    assert api_client.get("/observations?min_value=13").json() == []
    assert api_client.post("/observations", json={"title":"x", "value":-1}).status_code == 422
    print(created.json())

## Unit 5.2: Endpoint tests and deployment — CO5

Tests should assert status codes, response fields, persistence and error cases. FastAPI TestClient runs the ASGI app without a network server. This repository uses an isolated temporary SQLite database per test. Django tests check validation, ownership, permissions, CSV atomicity, authentication and rendered pages. Deployment needs a production server, private database volume, secret configuration, HTTPS and static serving. The supplied container and guide are a reproducible starting point; a successful local run does not prove a hosted deployment.

1. Run the Django tests and FastAPI pytest suite.
2. Read tests for 201 creation, 422 validation and 404 missing records.
3. Run check, migration drift check and collectstatic.
4. Build the Docker image and run Django using the documented environment variables.
5. Follow docs/deployment.md for HTTPS, static serving, volumes and a release checklist.

Browser practical: `/course/`

Study commands (run shell examples in a terminal):
```text
python manage.py test
python -m pytest api/tests.py -q
python manage.py check
python manage.py makemigrations --check --dry-run
python manage.py collectstatic --noinput
docker build -t django-vlab .
```

Record your actual observation and explain one failure case before continuing.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "api/tests.py", "-q"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

## Completion record

For each unit record: input, observed output, invalid case, explanation and source file. The checks above exercise representative behavior; run the complete Django tests separately. Deployment and remote Git exercises require your terminal and infrastructure. Do not write a successful hosted deployment result unless you actually verified it.

In [ ]:
from django.db import connections
connections.close_all()
workspace.cleanup()
print("Temporary notebook data removed")